# Final basketball tracking from GitHub
Run numbered cells. Cell 3 alone starts GPU inference. Use a fresh Colab T4 runtime.


## Cell 1 — familiar ZIP + Drive setup


In [ ]:
from pathlib import Path
from google.colab import drive
import hashlib,json,os,shutil,subprocess,sys
import torch
REPO_URL='https://github.com/YuvanJain/sports-vision-tracking.git'
CHECKOUT=Path('/content/sports-vision-tracking')
ROOT=CHECKOUT/'reference/basketball/submission_stage1e'
DRIVE_ROOT=Path('/content/drive/MyDrive/Sports Vision Tracking')
VIDEO=DRIVE_ROOT/'Data/Basketball.mp4' # Edit for another compatible 30 FPS source clip.
WEIGHTS=DRIVE_ROOT/'yolov8s.pt'
OUTPUTS=DRIVE_ROOT/'outputs/basketball'
assert torch.cuda.is_available() and 'T4' in torch.cuda.get_device_name(0),'Select a T4 runtime.'
assert not CHECKOUT.exists(),'Use a fresh runtime; checkout exists.'
drive.mount('/content/drive')
assert VIDEO.is_file() and WEIGHTS.is_file(),'Drive video or existing yolov8s.pt missing.'
subprocess.run(['git','clone','--depth','1',REPO_URL,str(CHECKOUT)],check=True)
assert ROOT.is_dir(),'Submission code not found in cloned repository.'
manifest=json.loads((ROOT/'source_manifest.json').read_text())
for name,expected in manifest['sha256'].items():
    assert hashlib.sha256((ROOT/name).read_bytes()).hexdigest()==expected,name
(ROOT/'Data').mkdir(exist_ok=True)
(ROOT/'Data/Basketball.mp4').symlink_to(VIDEO)
(ROOT/'yolov8s.pt').symlink_to(WEIGHTS)
OUTPUTS.mkdir(parents=True,exist_ok=True)
os.chdir(ROOT)
installed=subprocess.run([sys.executable,'-m','pip','install','-r',str(ROOT/'requirements.txt')],text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(installed.stdout)
assert installed.returncode==0,'Install failed; stop before inference.'
print('Setup complete; no inference started.')


## Cell 2 — input, GPU and storage preflight


In [ ]:
import cv2,yaml,inspect
PREFLIGHT_OK=False
assert torch.cuda.is_available() and 'T4' in torch.cuda.get_device_name(0)
cap=cv2.VideoCapture(str(VIDEO))
try:
    assert cap.isOpened(),'Cannot open Basketball.mp4 from Drive.'
    fps=cap.get(cv2.CAP_PROP_FPS);count=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    size=(int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)))
finally:cap.release()
assert abs(fps-30)<.01 and count>=18900 and size==(3840,2160),(fps,count,size)
cfg=yaml.safe_load((ROOT/'configs/basketball_config.yaml').read_text())
assert cfg['final']=={'start_frame':900,'end_frame_exclusive':18900,'chunk_frames':3600,'chunk_count':5}
assert (ROOT/cfg['model']['weights']).resolve(strict=True)==WEIGHTS.resolve(strict=True)
assert WEIGHTS.stat().st_size>0 and os.access(OUTPUTS,os.W_OK)
local_gib=shutil.disk_usage('/content').free/2**30
drive_gib=shutil.disk_usage(OUTPUTS).free/2**30
print(f'Free local {local_gib:.1f} GiB; Drive mount reports {drive_gib:.1f} GiB. Confirm actual Drive quota separately.')
assert local_gib>=2,'Less than 2 GiB local runtime space.'
assert drive_gib>=8,'Less than 8 GiB reported free Drive space; stop without deleting files.'
sys.path.insert(0,str(ROOT))
from scripts.track_basketball_final import run
from scripts.verify_final import verify
assert list(inspect.signature(run).parameters)==['video','output','config','device','start_frame','end_frame']
PREFLIGHT_OK=True
print('Preflight passed. Cell 3 alone starts the 18000-frame GPU job.')


## Cell 3 — EXPLICIT full 10-minute GPU run


In [ ]:
from datetime import datetime,timezone
assert PREFLIGHT_OK,'Run Cell 2 first.'
assert 'RUN' not in globals(),'This cell has already started. Preserve partial output and use a fresh runtime.'
RUN=OUTPUTS/('final_stage1e_'+datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f'))
assert not RUN.exists()
command=[sys.executable,'-u',str(ROOT/'scripts/track_basketball_final.py'),'--video',str(VIDEO),'--output-dir',str(RUN),'--device','cuda:0']
log=OUTPUTS/(RUN.name+'.log')
with log.open('x') as f:
    child=subprocess.Popen(command,cwd=ROOT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in child.stdout:print(line,end='',flush=True);f.write(line);f.flush()
    rc=child.wait()
assert rc==0,f'Run stopped (exit {rc}). Preserve {RUN} and inspect {log}. Restart from frame900 in a NEW folder; IDs cannot resume unchanged.'
print('GPU run finished:',RUN)


## Cell 4 — read-only five-chunk export verification


In [ ]:
assert 'RUN' in globals() and RUN.is_dir(),'Set RUN to the completed output folder first.'
from scripts.verify_final import verify
result=verify(RUN)
print(json.dumps(result,indent=2))
print('Inspect all five annotated MP4s visually. Shot outcomes were not implemented.')
